# Association Rule Mining

This is a sample solution for the association rule mining exercise. This does not mean that this is the only way to solve this exercise. As with any programming task - and also with most data analysis tasks - there are multiple solutions for the same problem. 

## Libraries and Data

The first part of the exercise is about association rule mining. In Python, you can use the ```mlxtend``` library for the mining of association rules. 

We use data about [store baskets](https://sherbold.github.io/intro-to-data-science/exercises/data/store_data.csv) in this exercise. You can use the following code to load the data. The code creates a list of records, where each record is a list of the items that are part of the transaction.

In [2]:
with open('data/store_data.csv') as f:
    records = []
    for line in f:
        records.append(line.strip().split(','))

## Finding frequent itemsets

Once you have the transactional records, use the apriori algorithm to find frequent itemsets with a suitable threshold for support for this data. Try to find a suitable threshold for the minimal support such that you can state a clear reason why you picked this threshold. 

In [3]:
import pandas as pd
from mlxtend.frequent_patterns import apriori
from mlxtend.preprocessing import TransactionEncoder

# we first need to create a one-hot encoding of our transactions
te = TransactionEncoder()
te_ary = te.fit_transform(records)
data_df = pd.DataFrame(te_ary, columns=te.columns_)

# use support of 0.005 - low threshold may include to many candidates
# careful selection of rules based on other metrics required
# this means that we use a higher confidence
frequent_itemsets = apriori(pd.DataFrame(
    data_df), min_support=0.005, use_colnames=True)
frequent_itemsets

,support,itemsets
0,0.020397,frozenset({almonds})
1,0.008932,frozenset({antioxydant juice})
2,0.033329,frozenset({avocado})
3,0.008666,frozenset({bacon})
4,0.010799,frozenset({barbecue sauce})
...,...,...
720,0.007466,"frozenset({spaghetti, soup, mineral water})"
721,0.009332,"frozenset({spaghetti, tomatoes, mineral water})"
722,0.006399,"frozenset({spaghetti, mineral water, turkey})"
723,0.006266,"frozenset({spaghetti, mineral water, whole whe..."


## Mining rules from the frequent itemsets

Determine good rules from the results for this data. Use lift and confidence as metrics for your evaluations. 

In [4]:
from mlxtend.frequent_patterns import association_rules

# use a high confidence to counterbalance the low support threshold
# order by lift to have the "best" rules at the top
association_rules(frequent_itemsets, metric="confidence",
                  min_threshold=0.5).sort_values('lift', ascending=False)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
13,"frozenset({shrimp, ground beef})",frozenset({spaghetti}),0.011465,0.174110,0.005999,0.523256,3.005315,1.0,0.004003,1.732354,0.674995,0.033408,0.422751,0.278856
6,"frozenset({frozen vegetables, ground beef})",frozenset({spaghetti}),0.016931,0.174110,0.008666,0.511811,2.939582,1.0,0.005718,1.691742,0.671179,0.047515,0.408893,0.280791
9,"frozenset({olive oil, frozen vegetables})",frozenset({spaghetti}),0.011332,0.174110,0.005733,0.505882,2.905531,1.0,0.003760,1.671444,0.663346,0.031899,0.401715,0.269404
8,"frozenset({soup, frozen vegetables})",frozenset({mineral water}),0.007999,0.238368,0.005066,0.633333,2.656954,1.0,0.003159,2.077178,0.628658,0.020994,0.518578,0.327293
17,"frozenset({olive oil, soup})",frozenset({mineral water}),0.008932,0.238368,0.005199,0.582090,2.441976,1.0,0.003070,1.822476,0.595818,0.021476,0.451296,0.301951
7,"frozenset({olive oil, frozen vegetables})",frozenset({mineral water}),0.011332,0.238368,0.006532,0.576471,2.418404,1.0,0.003831,1.798297,0.593226,0.026864,0.443918,0.301938
15,"frozenset({milk, soup})",frozenset({mineral water}),0.015198,0.238368,0.008532,0.561404,2.355194,1.0,0.004909,1.736520,0.584287,0.034820,0.424136,0.298599
2,"frozenset({soup, chocolate})",frozenset({mineral water}),0.010132,0.238368,0.005599,0.552632,2.318395,1.0,0.003184,1.702471,0.574488,0.023052,0.412618,0.288061
3,"frozenset({eggs, cooking oil})",frozenset({mineral water}),0.011732,0.238368,0.006399,0.545455,2.288286,1.0,0.003603,1.675590,0.569675,0.026258,0.403195,0.286150
5,"frozenset({frozen vegetables, ground beef})",frozenset({mineral water}),0.016931,0.238368,0.009199,0.543307,2.279277,1.0,0.005163,1.667711,0.570931,0.037378,0.400376,0.290949


The rules make sense, but are only helpful in a limited way, because we only have two different items as consequent: spaghetti and mineral water. The first three rules indicate that if a person buys a product that is often used to make pasta sauce (e.g., ground beef, vegetables, olive oil), they also often buy pasta. While this makes sense, this rule may also be there, because people tend to buy pasta a lot. However, the lift of about 3 indicates that these rules are three times as likely as a random effect. The shrimp does not really match this pattern. 

The other rules are likely effective, but an random artifact: the problem is that mineral water is part of many transactions:

In [5]:
# the mean of a one hot encoded column is the percentage that this value occurs
data_df['mineral water'].mean()

np.float64(0.23836821757099053)

Thus, the rule to predict mineral water is almost universally good. 

## Validation of the rules

Randomly split your records into two sets with roughly 50% of data each. Now use the Apriori algorithm to determine rules on both of these sets. Do you find similar rules on both sets? What does the similarity/the differences indicate?

In [6]:
from sklearn.model_selection import train_test_split

# split the data into two sets with 50% of the data
X_train, X_test = train_test_split(data_df, test_size=0.5, random_state=42)

# create frequent itemsets
frequent_itemsets_train = apriori(pd.DataFrame(
    X_train), min_support=0.005, use_colnames=True)
frequent_itemsets_test = apriori(pd.DataFrame(
    X_test), min_support=0.005, use_colnames=True)

In [7]:
# rules for first set
association_rules(frequent_itemsets_train, metric="confidence",
                  min_threshold=0.5).sort_values('lift', ascending=False)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
27,"frozenset({spaghetti, whole wheat pasta})",frozenset({milk}),0.010133,0.135733,0.005067,0.500000,3.683694,1.0,0.003691,1.728533,0.735991,0.035985,0.421475,0.268664
5,"frozenset({eggs, grated cheese})",frozenset({spaghetti}),0.009600,0.181333,0.005867,0.611111,3.370098,1.0,0.004126,2.105143,0.710090,0.031700,0.524973,0.321732
15,"frozenset({soup, frozen vegetables})",frozenset({spaghetti}),0.008533,0.181333,0.005067,0.593750,3.274357,1.0,0.003519,2.015179,0.700575,0.027417,0.503766,0.310846
10,"frozenset({frozen vegetables, ground beef})",frozenset({spaghetti}),0.016800,0.181333,0.009600,0.571429,3.151261,1.0,0.006554,1.910222,0.694331,0.050919,0.476501,0.312185
21,"frozenset({shrimp, ground beef})",frozenset({spaghetti}),0.012267,0.181333,0.006933,0.565217,3.117008,1.0,0.004709,1.882933,0.687614,0.037143,0.468914,0.301726
34,"frozenset({milk, frozen vegetables, mineral wa...",frozenset({spaghetti}),0.011200,0.181333,0.006133,0.547619,3.019958,1.0,0.004102,1.809684,0.676446,0.032904,0.447417,0.290721
14,"frozenset({olive oil, frozen vegetables})",frozenset({spaghetti}),0.011200,0.181333,0.005867,0.523810,2.888655,1.0,0.003836,1.719200,0.661224,0.031429,0.418334,0.278081
0,"frozenset({cake, chocolate})",frozenset({spaghetti}),0.013333,0.181333,0.006933,0.520000,2.867647,1.0,0.004516,1.705556,0.660083,0.036932,0.413681,0.279118
6,"frozenset({herb & pepper, eggs})",frozenset({spaghetti}),0.012800,0.181333,0.006400,0.500000,2.757353,1.0,0.004079,1.637333,0.645597,0.034091,0.389251,0.267647
32,"frozenset({spaghetti, eggs, milk})",frozenset({mineral water}),0.009067,0.249600,0.006133,0.676471,2.710219,1.0,0.003870,2.319418,0.636800,0.024287,0.568857,0.350522


In [8]:
# rules for second set
association_rules(frequent_itemsets_test, metric="confidence",
                  min_threshold=0.5).sort_values('lift', ascending=False)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
20,"frozenset({pancakes, olive oil})",frozenset({spaghetti}),0.010930,0.166889,0.005599,0.512195,3.069080,1.0,0.003774,1.707878,0.681620,0.032508,0.414478,0.272871
16,"frozenset({olive oil, tomatoes})",frozenset({mineral water}),0.007465,0.227139,0.005065,0.678571,2.987466,1.0,0.003370,2.404455,0.670272,0.022067,0.584105,0.350436
4,"frozenset({soup, chocolate})",frozenset({mineral water}),0.008798,0.227139,0.005865,0.666667,2.935055,1.0,0.003867,2.318582,0.665143,0.025492,0.568702,0.346244
15,"frozenset({olive oil, soup})",frozenset({mineral water}),0.008531,0.227139,0.005332,0.625000,2.751614,1.0,0.003394,2.060962,0.642054,0.023148,0.514790,0.324237
13,"frozenset({milk, turkey})",frozenset({mineral water}),0.011464,0.227139,0.006931,0.604651,2.662026,1.0,0.004328,1.954883,0.631587,0.029919,0.488460,0.317584
14,"frozenset({olive oil, shrimp})",frozenset({mineral water}),0.009064,0.227139,0.005332,0.588235,2.589754,1.0,0.003273,1.876947,0.619478,0.023095,0.467220,0.305855
12,"frozenset({milk, soup})",frozenset({mineral water}),0.014130,0.227139,0.008264,0.584906,2.575095,1.0,0.005055,1.861891,0.620431,0.035469,0.462912,0.310645
9,"frozenset({olive oil, frozen vegetables})",frozenset({mineral water}),0.011464,0.227139,0.006665,0.581395,2.559641,1.0,0.004061,1.846278,0.616386,0.028736,0.458370,0.305369
1,"frozenset({chocolate, chicken})",frozenset({mineral water}),0.012797,0.227139,0.007198,0.562500,2.476452,1.0,0.004291,1.766538,0.603925,0.030928,0.433921,0.297095
5,"frozenset({eggs, cooking oil})",frozenset({mineral water}),0.010930,0.227139,0.006132,0.560976,2.469741,1.0,0.003649,1.760405,0.601676,0.026437,0.431949,0.293985


The "mineral water" rules seem to be fairly stable and are present in both splits. The "spaghetti" rules seem to be more random. While there are such rules in both splits, they are not the same in the splits. In general, it seems like spaghetti may also be just something that is bought very often, same as mineral water, and that the rules may be good suggestions for cross-sell, but that there is probably not a strong association between the antecedents and the consequents for all rules, i.e., there does not seem to be causality. 